In [21]:
from google.colab import files
uploaded = files.upload()


Saving country_vaccinations.csv to country_vaccinations (2).csv


In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("country_vaccinations.csv")
df.head()

,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website
0,Afghanistan,AFG,2021-02-22,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
1,Afghanistan,AFG,2021-02-23,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
2,Afghanistan,AFG,2021-02-24,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
3,Afghanistan,AFG,2021-02-25,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
4,Afghanistan,AFG,2021-02-26,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/


Q1. Load the COVID-19 vaccination dataset. Create a
Series of total vaccinations per country. Examine the
vaccination disparity between the top 10 and bottom
10 countries using index operations.

In [23]:
#1

total_vaccinations = df.groupby("country")["total_vaccinations"].max()

top10 = total_vaccinations.nlargest(10)
bottom10 = total_vaccinations.nsmallest(10)

print("Top 10 Countries:\n", top10)
print("\nBottom 10 Countries:\n", bottom10)

Top 10 Countries:
 country
China            3.263129e+09
India            1.834501e+09
United States    5.601818e+08
Brazil           4.135596e+08
Indonesia        3.771089e+08
Japan            2.543456e+08
Bangladesh       2.436427e+08
Pakistan         2.193686e+08
Vietnam          2.031444e+08
Mexico           1.919079e+08
Name: total_vaccinations, dtype: float64

Bottom 10 Countries:
 country
Pitcairn                94.0
Tokelau               1936.0
Niue                  4161.0
Montserrat            4211.0
Falkland Islands      4407.0
Saint Helena          7892.0
Tuvalu               12114.0
Burundi              12166.0
Wallis and Futuna    13073.0
Nauru                16824.0
Name: total_vaccinations, dtype: float64


Created a Pandas Series to represent the total number of vaccinations for each country in order to analyze how vaccines are distributed globally.

Steps Performed:
First, I grouped the dataset based on the country column so that all records belonging to the same country are combined together. Then, I extracted the maximum value of total vaccinations for each country, since the dataset contains cumulative values over time. After that, I sorted the values in descending order to identify the top 10 countries with the highest vaccination counts and in ascending order to find the bottom 10 countries with the lowest counts. Finally, I used index-based operations to easily compare these extreme values.

Observations:
From the analysis, I observed that the top 10 countries have extremely high vaccination counts, which is mainly due to their large population size and strong healthcare systems. In contrast, the bottom 10 countries have very low vaccination numbers, which may be due to smaller population size, limited resources, or slower vaccine distribution.

This clearly highlights the inequality in global vaccination distribution. The use of sorting and indexing in Pandas makes it very easy to identify and compare such disparities across countries.

Q2. Examine the relationship between people_vaccinated and people_fully_vaccinated using ufuncs.

In [24]:
#2

df["log_vaccinated"] = np.log1p(df["people_vaccinated"])
df["sqrt_fully"] = np.sqrt(df["people_fully_vaccinated"])

print(df[["log_vaccinated", "sqrt_fully"]].corr())

                log_vaccinated  sqrt_fully
log_vaccinated         1.00000     0.75697
sqrt_fully             0.75697     1.00000


Analyzed the relationship between partially vaccinated and fully vaccinated people using mathematical transformations.

Steps Performed:
First, I applied a logarithmic transformation using np.log1p on the people_vaccinated column to reduce skewness in the data. Then, I applied a square root transformation on the people_fully_vaccinated column. After transforming the data, I calculated the correlation between these two columns to understand their relationship.

Observations:
The correlation value is positive, which indicates that as the number of vaccinated people increases, the number of fully vaccinated people also increases. However, the relationship is not perfectly proportional, which means some people may delay or skip the second dose. This shows that full vaccination depends on multiple real-world factors.

Q3. Apply hierarchical indexing and extract weekly vaccination data using xs().

In [25]:
#3

df["date"] = pd.to_datetime(df["date"])

multi_df = df.set_index(["country", "date"]).sort_index()

weekly = multi_df.groupby(level=0).resample("W", level=1).sum(numeric_only=True)

india_data = weekly.xs("India").head()
multi_country_data = weekly.loc[["India", "United States", "Brazil"]].head()

print("India Data:\n", india_data)
print("\nMultiple Countries Data:\n", multi_country_data)

India Data:
             total_vaccinations  people_vaccinated  people_fully_vaccinated  \
date                                                                         
2021-01-17            415482.0           415482.0                      0.0   
2021-01-24           7567199.0          7567199.0                      0.0   
2021-01-31          20340525.0         20340525.0                      0.0   
2021-02-07          34502604.0         34502604.0                      0.0   
2021-02-14          43412794.0         43405126.0                   7668.0   

            daily_vaccinations_raw  daily_vaccinations  \
date                                                     
2021-01-17                224301.0            303331.0   
2021-01-24               1391203.0           1251394.0   
2021-01-31               2143339.0           1824760.0   
2021-02-07               2053519.0           2023154.0   
2021-02-14               2240092.0           2456402.0   

            total_vaccinations_pe

Applied hierarchical indexing to analyze vaccination trends at multiple levels.

Steps Performed:
First, I converted the date column into datetime format for proper time-based operations. Then, I created a multi-level index using country and date. After that, I grouped the data by country and resampled it weekly to reduce daily fluctuations. Finally, I extracted specific country data using the xs() function.

Observations:
Hierarchical indexing makes it easier to analyze both country-wise and time-wise data. Weekly aggregation provides smoother trends compared to daily data. This approach is very useful for understanding long-term patterns in time-series datasets.

Q4. Create a Vaccination Efficiency Score and rank countries.

In [26]:
# Q4

eff = df.groupby("country")[["people_vaccinated","people_fully_vaccinated"]].max()

eff["efficiency_score"] = eff["people_fully_vaccinated"] / eff["people_vaccinated"]

print(eff.sort_values("efficiency_score", ascending=False).head())

                   people_vaccinated  people_fully_vaccinated  \
country                                                         
Tokelau                        968.0                    968.0   
Qatar                      2593517.0                2593517.0   
Pitcairn                        47.0                     47.0   
Togo                       1558542.0                1557538.0   
Wallis and Futuna             6483.0                   6457.0   

                   efficiency_score  
country                              
Tokelau                    1.000000  
Qatar                      1.000000  
Pitcairn                   1.000000  
Togo                       0.999356  
Wallis and Futuna          0.995990  


Calculated a vaccination efficiency score to evaluate how effectively countries complete vaccinations.

Steps Performed:
First, I grouped the dataset by country and extracted the maximum values of vaccinated and fully vaccinated people. Then, I calculated a ratio by dividing fully vaccinated people by total vaccinated people. After that, I sorted the values to rank countries based on efficiency.

Observations:
Countries with scores close to 1 are more efficient in completing vaccinations. Some countries have lower scores, indicating incomplete vaccination coverage. This metric provides deeper insight than just total vaccination counts.

Q5. Identify columns with more than 30% missing values and handle them.

In [27]:
# Q5

missing = df.isnull().mean() * 100
print(missing)

cols_to_drop = missing[missing > 30]
print("\nColumns with high missing values:\n", cols_to_drop)


df = df.sort_values(["country","date"])
df_ffill = df.fillna(method="ffill")

df_ffill.head()

country                                 0.000000
iso_code                                0.000000
date                                    0.000000
total_vaccinations                     49.594276
people_vaccinated                      52.267893
people_fully_vaccinated                55.148419
daily_vaccinations_raw                 59.124746
daily_vaccinations                      0.345617
total_vaccinations_per_hundred         49.594276
people_vaccinated_per_hundred          52.267893
people_fully_vaccinated_per_hundred    55.148419
daily_vaccinations_per_million          0.345617
vaccines                                0.000000
source_name                             0.000000
source_website                          0.000000
log_vaccinated                         52.267893
sqrt_fully                             55.148419
dtype: float64

Columns with high missing values:
 total_vaccinations                     49.594276
people_vaccinated                      52.267893
people_fully_vacci

/tmp/ipykernel_5972/4261350762.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = df.fillna(method="ffill")


,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website,log_vaccinated,sqrt_fully
0,Afghanistan,AFG,2021-02-22,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/,0.0,NaN
1,Afghanistan,AFG,2021-02-23,0.0,0.0,NaN,NaN,1367.0,0.0,0.0,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/,0.0,NaN
2,Afghanistan,AFG,2021-02-24,0.0,0.0,NaN,NaN,1367.0,0.0,0.0,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/,0.0,NaN
3,Afghanistan,AFG,2021-02-25,0.0,0.0,NaN,NaN,1367.0,0.0,0.0,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/,0.0,NaN
4,Afghanistan,AFG,2021-02-26,0.0,0.0,NaN,NaN,1367.0,0.0,0.0,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/,0.0,NaN


Analyzed missing data and applied appropriate handling techniques.

Steps Performed:
First, I calculated the percentage of missing values in each column. Then, I identified columns with more than 30% missing values. Instead of removing data, I sorted the dataset and applied forward fill to replace missing values using previous records.

Observations:
Many columns contain a high percentage of missing values. Dropping such data would result in major information loss. Forward fill is suitable because vaccination data is time-based and values change gradually.

Q6. Filter countries using AstraZeneca vaccine and compare with global average.

In [28]:
# Q6

az = df[df["vaccines"].str.contains("AstraZeneca", na=False)]

series = az.groupby("country")["total_vaccinations"].max()

global_avg = total_vaccinations.mean()

difference = series - global_avg

print(series - global_avg)

country
Afghanistan   -4.529581e+07
Albania       -4.829258e+07
Algeria       -3.734193e+07
Andorra       -5.089483e+07
Angola        -3.351141e+07
                   ...     
Vietnam        1.520975e+08
Wales         -4.411939e+07
Yemen         -5.023932e+07
Zambia        -4.764421e+07
Zimbabwe      -4.200710e+07
Name: total_vaccinations, Length: 183, dtype: float64


Analyzed vaccination performance of countries using AstraZeneca vaccine.

Steps Performed:
First, I filtered the dataset to include only countries using AstraZeneca. Then, I grouped the data by country and extracted total vaccinations. After that, I calculated the global average and compared each country’s value with it.

Observations:
Some countries perform above the global average while others are below. This shows that vaccine type alone does not determine performance, and other factors like infrastructure also play a role.

Q7. Calculate daily vaccination rate using diff().

In [29]:
# Q7

df["daily_rate"] = df.groupby("country")["total_vaccinations"].diff()

top_daily = df.groupby("country")["daily_rate"].max().sort_values(ascending=False)

print(top_daily.head())

country
China         24741000.0
India         18627269.0
Mexico         7246123.0
Bangladesh     7038358.0
Japan          6586453.0
Name: daily_rate, dtype: float64


Calculated daily vaccination rate to measure speed of vaccination.

Steps Performed:
First, I grouped the dataset by country. Then, I used the diff() function to calculate the difference between consecutive values. After that, I identified the maximum daily vaccination rates for each country.

Observations:
Higher values indicate strong vaccination drives. Sudden spikes suggest mass vaccination campaigns. This helps in identifying top-performing countries.

Q8. Compare dropna() and forward fill for India data.

In [30]:
# Q8

india = df[df["country"]=="India"]

india_drop = india.dropna()
india_ffill = india.ffill()

print(india.describe())
print(india_drop.describe())
print(india_ffill.describe())

                                date  total_vaccinations  people_vaccinated  \
count                            439        4.280000e+02       4.280000e+02   
mean   2021-08-21 23:59:59.999999744        7.557016e+08       4.808872e+08   
min              2021-01-15 00:00:00        0.000000e+00       0.000000e+00   
25%              2021-05-04 12:00:00        1.619355e+08       1.307268e+08   
50%              2021-08-22 00:00:00        6.207026e+08       4.797859e+08   
75%              2021-12-09 12:00:00        1.332416e+09       8.170081e+08   
max              2022-03-29 00:00:00        1.834501e+09       9.848381e+08   
std                              NaN        6.315688e+08       3.533168e+08   

       people_fully_vaccinated  daily_vaccinations_raw  daily_vaccinations  \
count             3.990000e+02            4.170000e+02        4.380000e+02   
mean              2.918207e+08            4.233133e+06        4.175994e+06   
min               7.668000e+03            5.671000e+03

Compared two methods for handling missing data.

Steps Performed:
First, I filtered data for India. Then, I applied dropna() to remove missing values and ffill() to fill them. Finally, I compared summary statistics of both methods.

Observations:
dropna() reduces dataset size significantly, while ffill() preserves data continuity. Forward fill is more suitable for time-series analysis.

Q9. Extract records where daily vaccinations per million exceed 10,000.

In [31]:
# Q9

high = df[df["daily_vaccinations_per_million"] > 10000]

print(high[["country","vaccines"]].drop_duplicates())

                   country                                           vaccines
1318               Andorra       Moderna, Oxford/AstraZeneca, Pfizer/BioNTech
2051              Anguilla                Oxford/AstraZeneca, Pfizer/BioNTech
2553   Antigua and Barbuda     Oxford/AstraZeneca, Pfizer/BioNTech, Sputnik V
3653                 Aruba                                    Pfizer/BioNTech
4200             Australia       Moderna, Oxford/AstraZeneca, Pfizer/BioNTech
...                    ...                                                ...
83359           Uzbekistan  Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, ...
84034            Venezuela  Abdala, Sinopharm/Beijing, Sinovac, Soberana02...
84439              Vietnam  Abdala, Moderna, Oxford/AstraZeneca, Pfizer/Bi...
84697                Wales       Moderna, Oxford/AstraZeneca, Pfizer/BioNTech
85075    Wallis and Futuna                                            Moderna

[133 rows x 2 columns]


Identified countries with high vaccination rates using boolean indexing.

Steps Performed:
First, I filtered the dataset based on daily vaccinations per million greater than 10,000. Then, I extracted unique country and vaccine combinations.

Observations:
Countries with high vaccination rates often use multiple vaccines and have strong healthcare systems. This enables faster rollout.

Q10. Create pivot table for multi-country comparison.

In [32]:
# Q10

pivot = pd.pivot_table(
    df,
    values="total_vaccinations",
    index="country",
    columns="vaccines",
    aggfunc="max"
)

pivot.to_csv("vaccination_summary.csv")

pivot.head()

vaccines,"Abdala, Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Soberana02, Sputnik Light, Sputnik V","Abdala, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V","Abdala, Sinopharm/Beijing, Sinovac, Soberana02, Sputnik Light, Sputnik V","Abdala, Soberana Plus, Soberana02","COVIran Barekat, Covaxin, FAKHRAVAC, Oxford/AstraZeneca, Razi Cov Pars, Sinopharm/Beijing, Soberana02, SpikoGen, Sputnik V","CanSino, Covaxin, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac, Sputnik V","CanSino, Johnson&Johnson, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinovac, Sputnik V","CanSino, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V","CanSino, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sinovac","CanSino, Oxford/AstraZeneca, Pfizer/BioNTech, Sinovac",...,"Oxford/AstraZeneca, Sputnik V",Pfizer/BioNTech,"Pfizer/BioNTech, Sinopharm/Beijing","Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V","Pfizer/BioNTech, Sinovac","Pfizer/BioNTech, Sinovac, Turkovac","Pfizer/BioNTech, Sputnik V","QazVac, Sinopharm/Beijing, Sputnik V",Sinopharm/Beijing,"Sinopharm/Beijing, Sputnik V"
country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Algeria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Angola,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Created a pivot table to compare vaccination data across countries and vaccine types.

Steps Performed:
First, I used pivot_table() to restructure the dataset. Then, I set country as index and vaccines as columns. Finally, I aggregated values using the maximum function and exported the result.

Observations:
Pivot tables provide a clear and structured comparison. This makes it easier to analyze vaccine usage across countries and is useful for reporting.